# 📉➡️📈 BIST Derin Değer & Kontraryan Dip Avcısı

**Momentum kovalamayan, "aşırı ucuz kalmış ama çürük olmayan" hisseyi avlayan tarama.**

> *30 yıllık BIST tüccarı mantığı: "Malı ucuza almak kazandırır — ama sadece mal sağlamsa.
> Ucuz + çürük = değer tuzağı. Bütün maharet, ucuz-ve-sağlam ile ucuz-çünkü-batıyor'u ayırmakta."*

---

## Neden bu strateji? (Felsefe)

Bu depodaki mevcut bot (`src/scanner.py`, `src/model.py`) bir **momentum/trend-takip**
sistemidir: ADX>20, EMA hizası, yukarı kırılım, XGBoost yön tahmini. Yani **yükselişi kovalar**.

Bu notebook bunun **tam tersi** felsefeyi kurgular:

| | Momentum botu (mevcut) | Bu tarama (kontraryan derin değer) |
|---|---|---|
| Ne arar? | Yükselen, ivmeli hisse | Dövülmüş, aşırı satımda, terk edilmiş hisse |
| Ne zaman alır? | Kırılımda, kalabalıkla | Panikte, kalabalığın tersine, kademeli |
| Risk | Zirveye yakın alım | "Düşen bıçak" (kademeli alım + stop ile yönetilir) |
| Kâr mantığı | Trend devam eder | Ortalamaya dönüş + değerin fiyatı yakalaması |

**Ana belirleyici = TEKNİK göstergeler.** Bir hissenin ne kadar "aşırı ucuz" olduğunu teknik
ölçer ve sıralar. **Banker (akıllı para) + temel göstergeler ise ÜSTÜNE ekstra puanlama**
katmanıdır — çürük malı eler, sağlam olanı öne çıkarır. Alım tarafında ise **Fibonacci ile
3 kademeli alım** planı kurulur.


## 🏗️ Mimari — nasıl puanlıyoruz?

```
                      ┌──────────────────────────────────────────┐
   GÜNLÜK OHLCV  ───► │  EKSEN A: TEKNİK UCUZLUK (0-100)  [ANA]   │ ──► sıralamanın çıpası
   (tvdatafeed/       │  52h dip konumu·RSI(g+h)·drawdown·        │     + "aşırı ucuz" KAPISI
    borsapy/yf)       │  Bollinger %B·200EMA sapması·Williams%R    │     (TECH_GATE = 55)
                      └──────────────────────────────────────────┘
                                        ×
                      ┌──────────────────────────────────────────┐
                      │  EKSTRA PUANLAMA → çarpan bonus (±%35)     │
   OHLCV+Hacim  ───►  │   • BANKER / akıllı para (CMF·MFI·A-D·OBV· │
                      │     pozitif diverjans = birikim)          │
   İş Yatırım / yf ►  │   • TEMEL değer&kalite (PD/DD·EV/EBITDA·   │
                      │     F/K·DCF güvenlik marjı·owner earnings) │
                      └──────────────────────────────────────────┘
                                        ×
                      ┌──────────────────────────────────────────┐
   Temel + OHLCV ──►  │  VALUE-TRAP CEZASI (çarpan / diskalifiye)  │
                      │   nakit yakma·zarar·DCF pahalı·likidite·   │
                      │   "ucuz çünkü batıyor"·düşen bıçak         │
                      └──────────────────────────────────────────┘
                                        ↓
             NİHAİ SKOR = Teknik × trap_çarpanı × bonus_çarpanı   (yüksek = al)
                                        ↓
                    Seçilenlere → FIBONACCI 3-KADEME ALIM MERDİVENİ
```

**Neden çarpımsal?** Bir hisse **hem** teknik olarak aşırı ucuz **hem** sağlam olmalı.
Teknik ana çıpadır; banker+temel onu ±%35 oynatır; tuzak bayrağı ise ağır ceza (0.3×–0.8×)
ya da tam diskalifiye getirir. Böylece en çok dövülmüş hisse otomatik "al" olmaz —
**neden ucuz olduğu** sorgulanır.


## 1) Kurulum ve modüller

Bu notebook depodaki `src/deep_value.py` (saf puanlama motoru) ve `src/deep_value_data.py`
(veri adapteri) modüllerini kullanır. Colab'da deposu klonlayıp bu notebook'u kök dizinde
çalıştırın. Veri kaynağı zinciri: **tvdatafeed (rongardF) → yfinance**; temel oranlar:
**İş Yatırım → yfinance**.

In [ ]:
# Colab / yerel kurulum
import sys, subprocess, os

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import pandas, numpy, matplotlib, requests  # noqa
except ImportError:
    _pip("pandas", "numpy", "matplotlib", "requests")

# tvdatafeed (rongardF fork) — TradingView verisi; yoksa yfinance yedeğe düşülür
try:
    import tvDatafeed  # noqa
except ImportError:
    _pip("git+https://github.com/rongardF/tvdatafeed.git", "yfinance")

# Depo kökünü sys.path'e ekle (notebook repo kökünde çalışır)
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import importlib
from src import deep_value as dv
from src import deep_value_data as dvd
importlib.reload(dv); importlib.reload(dvd)
print("Motor yüklendi. TECH_GATE =", dv.TECH_GATE, "| BONUS_STRENGTH =", dv.BONUS_STRENGTH)

## 2) EKSEN A — Teknik Ucuzluk (ANA belirleyici, 0-100)

100 = maksimum dövülmüş / aşırı satımda. Trend-takibin **tersine**: düşük RSI, dipteki konum,
derin drawdown **yüksek** puan alır. Alt bileşenler ve ağırlıkları:

| Bileşen | Ağırlık | Mantık |
|---|---|---|
| **52-hafta banttaki konum** | %28 | Dibe yakınlık — "aşırı ucuz"un kalbi. `(fiyat−52hDip)/(52hTepe−52hDip)` |
| **Günlük RSI(14)** | %20 | Aşırı satım (RSI 50→0 puan, ≤15→100) |
| **Haftalık RSI(14)** | %12 | Kalıcı ucuzluk teyidi (gürültüye dayanıklı) |
| **52-hafta zirveden düşüş** | %15 | Ne kadar dövülmüş (%10→0, %60→100) |
| **Bollinger %B** | %10 | Alt banda/altına sarkma |
| **200-EMA sapması** | %10 | Uzun vade ortalamanın ne kadar altında |
| **Williams %R** | %5 | Ek aşırı-satım teyidi |

Ayrıca **dip teyidi**: MACD histogramı son barlarda yukarı dönüyorsa (düşüş ivmesi kırıldı)
`dip_confirm=True`. Kapı: `teknik ≥ 55` → "aşırı ucuz aday".


## 3) BANKER / Akıllı Para katmanı (ekstra puan)

Aşırı ucuz bir hissede asıl aradığımız, fiyat dibe yatarken **güçlü ellerin sessizce
topladığının** izidir. Klasik dip-avı sinyali **pozitif diverjans**tır: fiyat düşük/yatay
ama para akışı yukarı.

- **CMF (Chaikin Money Flow)** > 0 → birikim (banker topluyor)
- **MFI (Money Flow Index)** → hacim ağırlıklı aşırı-satım/dönüş
- **A/D çizgisi & OBV eğimi** → kümülatif hacim baskısı yönü
- **Pozitif diverjans bonusu** → fiyat son 20 günde düşmüş **ama** A/D/CMF yukarı

Banker skoru **düşükse** hisse hâlâ *dağıtım* (satış) altındadır → "ucuz ama akıllı para
girmemiş, acele etme / daha derin kademeleri bekle". Yüksekse teknik ucuzluk daha güvenilir.


## 4) EKSEN B — Temel Değer & Kalite (ekstra puan) + Value-Trap elemesi

İki alt blok, **geometrik** birleştirilir (ikisi de gerekli):

- **Ucuzluk (value):** PD/DD, EV/EBITDA, F/K, EV/Satış — sektör-göreli ve mutlak.
  *Eksiğe dayanıklı*: BIST'te İş Yatırım bazı oranları vermez, Yahoo trailing F/K çevrimsel
  diplerde 500+ olabilir → eldeki metriklerin ağırlıklı ortalaması alınır, tek metrik skoru
  sıfırlamaz.
- **Kalite (quality):** owner earnings (gerçek nakit üretimi), DCF güvenlik marjı, Buffett
  skoru, borçluluk, ROE.

**Value-trap bayrakları** (30 yıllık trader refleksi) nihai skoru çarpanla cezalandırır:

| Bayrak | Ceza | Örnek (2026-07-03) |
|---|---|---|
| Nakit yakıyor (negatif owner earnings) | ×0.30 | ECILC (OE −1.305M) |
| Zarar (negatif net kâr) | ×0.55 | ECILC (−924M) |
| DCF'e göre pahalı + AVOID | ×0.55 | EREGL (marj −%1175), ARCLK (−%127) |
| Ucuz çünkü zarar (F/K yok + düşük PD/DD + nakit yok) | ×0.50 | CANTE (PD/DD 0.4, OE=NaN) |
| İşletme değeri pahalı (EV/EBITDA>15) | ×0.75 | SASA (EV/EBITDA 26) |
| Likidite tuzağı / düşen bıçak | ×0.60 / ×0.80 | — |

Nakit yakan **+** zarar eden **+** AVOID kombinasyonu → **tam diskalifiye**.


## 5) Alım tarafı — Fibonacci 3-Kademeli Merdiven

"Düşen bıçağı" tek hamlede tutmayız. Seçilen her hisse için son ~180 barın baskın swing'i
(tepe **H**, dip **L**) alınır; H→L düşüşünün Fibonacci seviyeleri referanstır. Alım
kademeleri **yalnızca güncel fiyatın altındaki** seviyelere konur ve **derine daha çok
ağırlık** verilir (ucuzsa daha çok al):

- **1. kademe (%25)** — güncel fiyat / ilk Fib desteği (0.618–1.0)
- **2. kademe (%35)** — 1.272 uzantısı (kapitülasyon)
- **3. kademe (%40)** — 1.618 uzantısı (aşırı kapitülasyon)
- **HARD STOP** = en derin kademe − 1.5×ATR (bu da tutmazsa tez yanlış)

DCF içsel değeri varsa merdivenin ağırlıklı ortalama maliyetine göre **beklenen getiri**
hesaplanır (teknik giriş ↔ temel hedef köprüsü).


## 6) Universe ve tarama — **TÜM BIST**

Evren, `src.scanner.fetch_bist_symbols()` ile **TradingView Scanner API'sinden tüm BIST
hisseleri** (canlı, ~500+ sembol) olarak çekilir; API'ye ulaşılamazsa HTML tablo yedeklerine,
en sonda `config.BIST100_SYMBOLS` çekirdek listesine düşer.

**Ana belirleyici teknik olduğu için** önce her hissenin teknik ucuzluğu hesaplanır; kapıyı
(TECH_GATE) geçemeyecek kadar pahalı olanlara **temel/banker verisi çekilmez** — yüzlerce
hissede gereksiz API isteği yapılmaz, tarama makul sürede biter. Yine de tüm evren teknik
olarak taranır.

In [ ]:
from src import config
from src.scanner import fetch_bist_symbols

# TÜM BIST evreni (canlı; başarısızsa yedeklere düşer)
SYMBOLS = fetch_bist_symbols()
print(f"Evren: {len(SYMBOLS)} BIST hissesi taranacak.")
# Hızlı deneme için küçük alt küme kullanmak isterseniz:
# SYMBOLS = ["ULKER","PETKM","TURSG","MAVI","FROTO","TUKAS","SASA","CANTE","ECILC","EREGL","ARCLK","GARAN"]

report, results = dvd.screen_universe(
    SYMBOLS,
    n_bars=600,           # ~2.5 yıl günlük bar (200EMA + 52h için yeterli)
    fib_lookback=180,     # Fibonacci swing penceresi
    tech_prefilter=True,  # teknik kapıyı geçemeyene temel veri çekme (tüm BIST için şart)
    with_ladder_top_n=20, # en iyi 20 adaya Fib merdiveni hesapla
)
print(f"\n{len(results)} hisse tarandı, {int(report['asiri_ucuz'].sum())} tanesi teknik kapıyı geçti.\n")
report.head(30)

### 6a) Sadece "aşırı ucuz" adaylar (kapıyı geçenler, tuzaksızlar önde)

In [ ]:
import pandas as pd
pd.set_option("display.width", 220, "display.max_columns", 30)

adaylar = report[report["asiri_ucuz"] & (~report["disqualified"])].copy()
cols = ["symbol","sector","final_score","tech_ucuzluk","banker","temel_skor","extra",
        "value","quality","rsi_d","rsi_w","pos_52w","drawdown","birikim","trap_mult","trap_flags"]
adaylar[cols].reset_index(drop=True)

### 6b) Değer tuzağı olarak elenenler (ders niteliğinde)

Bunlar teknik olarak ucuz **ama** temelde çürük — sistem bilerek geri iter. En çok dövülmüş
hissenin en iyi alım OLMADIĞINI gösterir.

In [ ]:
tuzaklar = report[(report["trap_flags"] != "") | (report["disqualified"])].copy()
tuzaklar[["symbol","final_score","tech_ucuzluk","temel_skor","value","quality",
          "trap_mult","disqualified","trap_flags"]].reset_index(drop=True)

## 7) Seçilen hisse için Fibonacci 3-kademe alım grafiği

In [ ]:
import matplotlib.pyplot as plt

def plot_fib_ladder(symbol, results, n_bars=250):
    res = next((r for r in results if r.symbol == symbol), None)
    if res is None or res.ladder is None:
        print(f"{symbol}: sonuç/merdiven yok."); return
    df = dvd.load_daily_ohlcv(symbol, n_bars=600).tail(n_bars)
    L = res.ladder
    fig, ax = plt.subplots(figsize=(13, 6))
    ax.plot(df.index, df["close"], color="#1f2d3d", lw=1.3, label="Kapanış")
    ax.axhline(L.swing_high, color="#888", ls="--", lw=0.8)
    ax.text(df.index[0], L.swing_high, f"  swing tepe {L.swing_high}", va="bottom", color="#888", fontsize=8)
    palette = ["#2e7d32", "#f9a825", "#c62828"]
    for i, rung in enumerate(L.rungs):
        c = palette[min(i, 2)]
        ax.axhline(rung["price"], color=c, lw=1.4, alpha=0.9)
        ax.text(df.index[-1], rung["price"],
                f"  {i+1}. kademe %{rung['weight_pct']} @ {rung['price']}",
                va="center", color=c, fontsize=9, fontweight="bold")
    ax.axhline(L.hard_stop, color="#6a1b9a", ls=":", lw=1.2)
    ax.text(df.index[-1], L.hard_stop, f"  STOP {L.hard_stop}", va="center", color="#6a1b9a", fontsize=8)
    title = (f"{symbol} — Nihai {res.final_score} | Teknik {res.technical.total} | "
             f"Banker {res.banker.total} | Temel {res.fundamental.total}")
    if L.expected_upside_pct is not None:
        title += f"\nDCF hedef {L.dcf_target} → merdiven ort. maliyetine göre bek. getiri %{L.expected_upside_pct}"
    ax.set_title(title, fontsize=11)
    ax.legend(loc="upper right"); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()

# En iyi aday için çiz:
en_iyi = report[report["asiri_ucuz"] & (~report["disqualified"])]
if not en_iyi.empty:
    plot_fib_ladder(en_iyi.iloc[0]["symbol"], results)

In [ ]:
def alim_plani(symbol, results, sermaye=100_000):
    """Bir hisse için kademeli alım planını TL bazında yazdırır."""
    res = next((r for r in results if r.symbol == symbol), None)
    if res is None or res.ladder is None:
        print(f"{symbol}: merdiven yok."); return
    L = res.ladder
    print(f"=== {symbol} — Kademeli Alım Planı (sermaye {sermaye:,.0f} TL) ===")
    print(f"Nihai skor {res.final_score} | Teknik {res.technical.total} | "
          f"Banker {res.banker.total} | Temel {res.fundamental.total}")
    if res.trap.flags:
        print("⚠️  Tuzak bayrakları:", "; ".join(res.trap.flags))
    print(f"Güncel {L.current_price} | swing {L.swing_low}–{L.swing_high} | STOP {L.hard_stop}\n")
    toplam_lot = 0
    for i, r in enumerate(L.rungs, 1):
        pay = sermaye * r["weight_pct"] / 100
        lot = int(pay / r["price"])
        toplam_lot += lot
        print(f"  {i}. kademe @ {r['price']:>8}  (%{r['weight_pct']:<4} = {pay:>10,.0f} TL "
              f"→ ~{lot:,} lot)  [{r['note']}]")
    print(f"\n  Toplam ~{toplam_lot:,} lot | STOP {L.hard_stop} "
          f"(en derin kademeden ~%{(1-L.hard_stop/L.rungs[-1]['price'])*100:.1f} aşağıda)")
    if L.expected_upside_pct is not None:
        print(f"  DCF hedef {L.dcf_target} → beklenen getiri ~%{L.expected_upside_pct}")

if not en_iyi.empty:
    alim_plani(en_iyi.iloc[0]["symbol"], results)

## 8) 📸 Canlı örnek çıktı — 2026-07-03 (gerçek veri)

Aşağıdaki tablo, motorun **gerçek İş Yatırım temel verisi + gerçek OHLCV** üzerinde
o günkü çıktısıdır. Notebook'u çalıştırınca güncel tarih için yeniden hesaplanır.

### En güçlü aşırı-ucuz adaylar (kapıyı geçen, tuzaksız)

| # | Hisse | Nihai | Teknik | Banker | Temel | value/quality | RSI | Neden |
|---|---|---|---|---|---|---|---|---|
| 1 | **ULKER** | ~68 | 67.8 | 25.4 | 89.5 | 87/91 | 30.3 | 52h dibinde, DCF marjı %68, EV/EBITDA 4.3 — ama akıllı para henüz girmemiş (CMF −0.12) |
| 2 | **PETKM** | yüksek | ✔ | – | 83.5 | 93/76 | 34.0 | PD/DD 0.7, güvenlik marjı %34, nakit üretiyor |
| 3 | **TURSG** | yüksek | ✔ | – | 76.3 | 55/100 | 37.8 | STRONG_BUY, F/K 5.6, güvenlik marjı %64 |
| 4 | **MAVI** | yüksek | ✔ | – | 76.7 | 68/84 | 38.9 | EV/EBITDA 2.9, güvenlik marjı %42 |
| 5 | **FROTO** | orta-yük | ✔ | – | 64.9 | 65/65 | 36.3 | Güçlü hendek, F/K 8.5, ama marj dar %17 |

### Değer tuzağı olarak elenenler (teknik ucuz ama çürük)

| Hisse | RSI | Neden ELENDİ | Çarpan |
|---|---|---|---|
| **CANTE** | **27.4** (en oversold!) | Owner earnings YOK, PD/DD 0.4 → "ucuz çünkü zarar" | ×0.50 |
| **ECILC** | 39.7 | Nakit yakıyor (−1.305M) + zarar (−924M) + AVOID | **diskalifiye** |
| **EREGL** | 30.6 | DCF marjı −%1175 (kâr çökmüş, çevrimsel), AVOID | ×0.55 |
| **ARCLK** | – | DCF marjı −%127, düşük nakit getirisi, AVOID | ×0.55 |

> **Kilit gözlem:** En çok dövülmüş hisse (CANTE, RSI 27) **en iyi alım değil** — sistem onu
> tuzak olarak eler. Asıl alınacak, hem aşırı ucuz **hem** nakit üreten ULKER/PETKM/TURSG.

### ULKER — gerçek 3-kademe Fibonacci merdiveni (güncel 100.1 ₺)

| Kademe | Fiyat | Ağırlık | Fib | Not |
|---|---|---|---|---|
| 1 | **96.5** | %25 | 1.000 | swing dip retesti |
| 2 | **84.2** | %35 | 1.272 | kapitülasyon uzantısı |
| 3 | **68.6** | %40 | 1.618 | aşırı kapitülasyon |
| **STOP** | **64.3** | — | — | en derin kademe − 1.5×ATR |

*52h yüksek 141.7 / düşük 96.5, drawdown %29, MACD dip teyidi ✅. Banker skoru düşük olduğu
için "kademeli gir, tek seferde yükleme" mesajı verir.*


## 9) ⚠️ Risk yönetimi ve kullanım notları

1. **Kademeli gir, aşkla değil planla.** 3 kademe = maliyet ortalaması + yanlışsa kontrollü
   zarar. Banker skoru düşükken (akıllı para girmemişken) ilk kademeyi küçük tut, derin
   kademeleri bekle.
2. **STOP'a sadık kal.** En derin kademe − 1.5×ATR kırılırsa tez yanlıştır; değer tuzağına
   dönmüş olabilir. Zararı büyütme.
3. **Value-trap bayraklarını ciddiye al.** "Ucuz" bir hisse aylarca daha ucuzlayabilir.
   Nakit yakan / zarar eden / DCF'e göre pahalı isimlerden uzak dur (sistem zaten cezalandırır).
4. **Katalizör/likidite.** Derin değer, katalizör olmadan uzun sürebilir. Likidite tuzağı
   bayraklı (düşük TL hacim) isimlerde pozisyon küçük olsun.
5. **Temel veri gecikmeli/eksiktir.** İş Yatırım oranları çeyrekliktir; teyit için son bilanço
   ve KAP'a bak. F/K çevrimsel diplerde yanıltır → PD/DD ve EV/EBITDA'ya ağırlık ver.
6. **Bu bir yatırım tavsiyesi değildir.** Eğitim ve araştırma amaçlı bir tarama aracıdır.
   Kararı kendi analizinle ver.

---

### Parametre ayarları (`src/deep_value.py`)
`TECH_GATE` (aşırı-ucuz eşiği, vars. 55) · `BONUS_STRENGTH` (ekstra katman gücü, ±%35) ·
`EXTRA_W_BANKER`/`EXTRA_W_FUND` · teknik alt-ağırlıklar `TECH_WEIGHTS` · Fibonacci
`_FIB_RATIOS` ve kademe ağırlıkları. Kendi tarzına göre kalibre et.
